# ISA-LLM
Hhow does an LLM understand indirect speech acts?

## Imports

In [8]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json


## Constants

In [ ]:
MODEL = 'gpt-5.4-nano-2026-03-17'

SYSTEM_PROMPT = '''
Classify the Person B's response as 1 or 0. Return 1 if Person B's response can be understood as a yes; otherwise return 0."
'''

SYSTEM_PROMPT2 = '''
Classify the Person B's response as a value between 1 and 0. Return values close to 1 if Person B's response can be understood as a yes; Return values close to 0 if Person B's response can be understood as a no. Return values close to 0.5 if Person B's response can be understood as a neither yes nor no"
'''

qa_pair = '''
Person A: Would you like some ice-cream?
Person B: It is so hot outside.
'''

QAS = [
    {"id": "1101", "text": "Person A: Have you met Martin lately?; Person B: I have not seen him for ages."},
    {"id": "1201", "text": "Person A: Are you and Martin still good friends?; Person B: I have not seen him for ages."},
    {"id": "1102", "text": "Person A: Are you doing something right now?; Person B: I am sending out an application."},
    {"id": "1202", "text": "Person A: Are you looking for a job?; Person B: I am sending out an application."},
]

texts_to_classify = [
    {"id": "text_1", "text": "First text goes here"},
    {"id": "text_2", "text": "Second text goes here"},
    {"id": "text_3", "text": "Third text goes here"},
]


## Query LLM

In [5]:
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

### Version 1: binary

In [12]:
client = OpenAI()

text_to_classify = "Your text goes here"

response = client.responses.create(
    model=MODEL,  # or another model that supports structured outputs
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": qa_pair,
        },
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "binary_classification",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "label": {
                        "type": "integer",
                        "enum": [0, 1]
                    }
                },
                "required": ["label"],
                "additionalProperties": False
            },
        }
    },
)

result = json.loads(response.output_text)
print(result)
print(result["label"])

{'label': 0}
0


### Version 2: continuous

In [ ]:
client = OpenAI()


response = client.responses.create(
    model=MODEL,  # or another model that supports structured outputs
    input=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT2,
        },
        {
            "role": "user",
            "content": qa_pair,
        },
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "batch_scores",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "score": {
                        "type": "number",
                        "minimum": 0,
                        "maximum": 1
                    }
                },
                "required": ["score"],
                "additionalProperties": False
            },
        }
    },
)

result = json.loads(response.output_text)
print(result)
print(result["score"])

{'score': 0.2}
0.2
